In [ ]:
import numpy as np

dat=np.load('/home/maria/SelfStudyThesis/data/allen_natural_scenes_four_class_composite.npy',allow_pickle=True).item()
#print(dat)
neural_event_probs=dat['X']
#neural_event_probs shape (40064,118)
print(neural_event_probs.shape)
image_logits=np.load('/home/maria/SelfStudyThesis/data/google_vit-base-patch16-224_embeddings_logits.pkl',allow_pickle=True)
print(image_logits)
print(image_logits['natural_scenes'].shape)
#natural_scenes_logits shape (118,1000)
natural_scenes_logits=image_logits['natural_scenes']

(40064, 118)
{'natural_movie_one': array([[-2.4532654 , -0.09083418, -0.5349715 , ..., -1.5723907 ,
        -0.5999451 ,  2.3458946 ],
       [-2.3504632 , -1.264612  ,  0.04747102, ..., -1.5550821 ,
        -0.6722498 ,  1.0309643 ],
       [-2.3651378 , -1.3212479 , -0.06854293, ..., -1.6624714 ,
        -0.64045095,  1.0618968 ],
       ...,
       [-0.90006524, -0.6913519 , -0.14579508, ..., -1.7344925 ,
        -0.38396636,  1.0407058 ],
       [-1.4154906 , -0.8844572 , -0.30757812, ..., -2.0306048 ,
        -1.045593  ,  1.6329633 ],
       [-1.6140486 , -0.92387295, -0.6117126 , ..., -2.4285073 ,
        -0.4353803 ,  2.4260607 ]], shape=(900, 1000), dtype=float32), 'natural_movie_two': array([[-1.1841856e+00, -1.1132178e+00, -2.3208432e-01, ...,
        -2.2917802e+00, -1.0776930e-03,  1.4986821e+00],
       [-8.2124376e-01, -5.5590606e-01, -2.4781108e-01, ...,
        -2.2458150e+00, -4.5277318e-01,  1.7404238e+00],
       [-6.7955476e-01, -6.0730571e-01, -2.2965023e-01, ...,

In [13]:
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr

# ============================================================
# LOAD DATA
# ============================================================

dat = np.load(
    '/home/maria/SelfStudyThesis/data/'
    'allen_natural_scenes_four_class_composite.npy',
    allow_pickle=True
).item()

# Original: neurons x images = (40064, 118)
X = dat['X'].T.astype(np.float64)

image_logits = np.load(
    '/home/maria/SelfStudyThesis/data/'
    'google_vit-base-patch16-224_embeddings_logits.pkl',
    allow_pickle=True
)

# images x ImageNet classes = (118, 1000)
Y = image_logits['natural_scenes'].astype(np.float64)

print("Neural X:", X.shape)
print("ViT Y:   ", Y.shape)

assert X.shape[0] == Y.shape[0]

n = X.shape[0]


# ============================================================
# PARAMETERS
# ============================================================

N_COMPONENTS = 10


# ============================================================
# STORAGE
# ============================================================

Y_pred = np.zeros_like(Y)

ranks = []
top1 = []
top5 = []


# ============================================================
# LEAVE-ONE-IMAGE-OUT
# ============================================================

for test_idx in range(n):

    train_idx = np.arange(n) != test_idx

    X_train = X[train_idx]
    X_test  = X[[test_idx]]

    Y_train = Y[train_idx]

    # --------------------------------------------------------
    # IMPORTANT:
    # scalers are fitted INSIDE each LOO fold
    # --------------------------------------------------------

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train_z = x_scaler.fit_transform(X_train)
    X_test_z  = x_scaler.transform(X_test)

    Y_train_z = y_scaler.fit_transform(Y_train)

    # --------------------------------------------------------
    # Teacher -> student mapping
    #
    # neural population activity -> ViT logits
    # --------------------------------------------------------

    pls = PLSRegression(
        n_components=N_COMPONENTS,
        scale=False,
        max_iter=1000
    )

    pls.fit(X_train_z, Y_train_z)

    # predict held-out ViT representation
    y_pred_z = pls.predict(X_test_z)

    # convert back to original ViT-logit coordinates
    y_pred = y_scaler.inverse_transform(y_pred_z)[0]

    Y_pred[test_idx] = y_pred

    # --------------------------------------------------------
    # IMAGE RETRIEVAL
    #
    # Which actual ViT representation is closest to our
    # neural prediction?
    # --------------------------------------------------------

    similarities = cosine_similarity(
        y_pred.reshape(1, -1),
        Y
    )[0]

    order = np.argsort(similarities)[::-1]

    rank = np.where(order == test_idx)[0][0] + 1

    ranks.append(rank)
    top1.append(rank == 1)
    top5.append(rank <= 5)

    print(
        f"\rImage {test_idx+1:3d}/{n} | "
        f"rank={rank:3d}",
        end=""
    )


print("\n")


# ============================================================
# GLOBAL METRICS
# ============================================================

ranks = np.asarray(ranks)

print("=" * 70)
print("TEACHER -> STUDENT LOO RESULTS")
print("=" * 70)

print(f"Top-1 retrieval accuracy : {np.mean(top1):.4f}")
print(f"Top-5 retrieval accuracy : {np.mean(top5):.4f}")
print(f"Median retrieval rank    : {np.median(ranks):.1f}")
print(f"Mean retrieval rank      : {np.mean(ranks):.1f}")


# ============================================================
# REPRESENTATIONAL SIMILARITY
# ============================================================

# similarity structure between images

true_RSM = np.corrcoef(Y)
pred_RSM = np.corrcoef(Y_pred)

iu = np.triu_indices(n, k=1)

r_rsm, p_rsm = pearsonr(
    true_RSM[iu],
    pred_RSM[iu]
)

print()
print("Representational geometry")
print("-" * 70)
print(f"RSM correlation: r={r_rsm:.4f}, p={p_rsm:.3e}")


# ============================================================
# PER-IMAGE REPRESENTATION CORRELATION
# ============================================================

image_corrs = []

for i in range(n):

    r, _ = pearsonr(
        Y[i],
        Y_pred[i]
    )

    image_corrs.append(r)

image_corrs = np.asarray(image_corrs)

print()
print("Held-out representation reconstruction")
print("-" * 70)
print(f"Mean correlation   : {image_corrs.mean():.4f}")
print(f"Median correlation : {np.median(image_corrs):.4f}")


# ============================================================
# CHANCE LEVELS
# ============================================================

print()
print("Retrieval chance levels")
print("-" * 70)
print(f"Top-1 chance       : {1/n:.4f}")
print(f"Top-5 chance       : {5/n:.4f}")
print(f"Expected mean rank : {(n+1)/2:.1f}")

Neural X: (118, 40064)
ViT Y:    (118, 1000)
Image 118/118 | rank= 12

TEACHER -> STUDENT LOO RESULTS
Top-1 retrieval accuracy : 0.0085
Top-5 retrieval accuracy : 0.0763
Median retrieval rank    : 39.0
Mean retrieval rank      : 45.3

Representational geometry
----------------------------------------------------------------------
RSM correlation: r=0.1682, p=5.473e-45

Held-out representation reconstruction
----------------------------------------------------------------------
Mean correlation   : 0.4053
Median correlation : 0.3981

Retrieval chance levels
----------------------------------------------------------------------
Top-1 chance       : 0.0085
Top-5 chance       : 0.0424
Expected mean rank : 59.5


In [14]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import pearsonr

# ============================================================
# LOAD
# ============================================================

dat = np.load(
    '/home/maria/SelfStudyThesis/data/'
    'allen_natural_scenes_four_class_composite.npy',
    allow_pickle=True
).item()

# Neural responses:
# original = neurons x images
Y = dat['X'].T.astype(np.float32)       # (118, 40064)

image_logits = np.load(
    '/home/maria/SelfStudyThesis/data/'
    'google_vit-base-patch16-224_embeddings_logits.pkl',
    allow_pickle=True
)

X = image_logits['natural_scenes'].astype(np.float32)  # (118, 1000)

print("ViT:", X.shape)
print("Neural:", Y.shape)

assert X.shape[0] == Y.shape[0]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

n_images = len(X)


# ============================================================
# MODEL
# ============================================================

class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim, output_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.GELU(),

            nn.Linear(512, 256),
            nn.GELU(),

            nn.Linear(256, latent_dim),
            nn.GELU(),

            nn.Linear(latent_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# PARAMETERS
# ============================================================

LATENT_DIM = 64
NEURAL_PCS = 50

EPOCHS = 500
LR = 1e-3
WEIGHT_DECAY = 1e-4


# ============================================================
# STORAGE
# ============================================================

Y_pred = np.zeros_like(Y)


# ============================================================
# LEAVE ONE IMAGE OUT
# ============================================================

for test_idx in range(n_images):

    train_mask = np.arange(n_images) != test_idx

    X_train = X[train_mask]
    X_test = X[[test_idx]]

    Y_train = Y[train_mask]
    Y_test = Y[[test_idx]]

    # --------------------------------------------------------
    # STANDARDIZE ViT FEATURES
    # FIT ONLY ON TRAINING DATA
    # --------------------------------------------------------

    x_scaler = StandardScaler()

    X_train_z = x_scaler.fit_transform(X_train)
    X_test_z = x_scaler.transform(X_test)

    # --------------------------------------------------------
    # PCA OF NEURAL RESPONSES
    # FIT ONLY ON TRAINING DATA
    #
    # Instead of predicting 40,064 outputs directly,
    # predict coordinates in the observable neural space.
    # --------------------------------------------------------

    y_mean = Y_train.mean(axis=0, keepdims=True)

    Y_train_centered = Y_train - y_mean

    pca = PCA(n_components=NEURAL_PCS)

    Z_train = pca.fit_transform(Y_train_centered)

    # Standardize PC scores
    z_scaler = StandardScaler()

    Z_train_z = z_scaler.fit_transform(Z_train)

    # --------------------------------------------------------
    # TORCH
    # --------------------------------------------------------

    Xtr = torch.tensor(
        X_train_z,
        dtype=torch.float32,
        device=device
    )

    Ztr = torch.tensor(
        Z_train_z,
        dtype=torch.float32,
        device=device
    )

    Xte = torch.tensor(
        X_test_z,
        dtype=torch.float32,
        device=device
    )

    model = Encoder(
        input_dim=X.shape[1],
        latent_dim=LATENT_DIM,
        output_dim=NEURAL_PCS
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    loss_fn = nn.MSELoss()

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    for epoch in range(EPOCHS):

        optimizer.zero_grad()

        pred = model(Xtr)

        loss = loss_fn(pred, Ztr)

        loss.backward()
        optimizer.step()

    # --------------------------------------------------------
    # PREDICT HELD-OUT IMAGE
    # --------------------------------------------------------

    model.eval()

    with torch.no_grad():

        Z_pred_z = model(Xte).cpu().numpy()

    # undo standardization
    Z_pred = z_scaler.inverse_transform(Z_pred_z)

    # reconstruct 40,064-dimensional neural response
    Y_hat = pca.inverse_transform(Z_pred) + y_mean

    Y_pred[test_idx] = Y_hat[0]

    print(
        f"\rLOO {test_idx+1:3d}/{n_images}",
        end=""
    )


print("\nDone.")


# ============================================================
# EVALUATION 1:
# POPULATION PATTERN CORRELATION PER IMAGE
# ============================================================

image_r = []

for i in range(n_images):

    r, _ = pearsonr(
        Y[i],
        Y_pred[i]
    )

    image_r.append(r)

image_r = np.asarray(image_r)

print("\n" + "=" * 70)
print("HELD-OUT IMAGE ENCODING")
print("=" * 70)

print(f"Mean population r:   {image_r.mean():.4f}")
print(f"Median population r: {np.median(image_r):.4f}")


# ============================================================
# EVALUATION 2:
# NEURON-WISE CORRELATION ACROSS IMAGES
# ============================================================

neuron_r = np.zeros(Y.shape[1])

for j in range(Y.shape[1]):

    if np.std(Y[:, j]) > 0 and np.std(Y_pred[:, j]) > 0:

        neuron_r[j] = np.corrcoef(
            Y[:, j],
            Y_pred[:, j]
        )[0, 1]

    else:
        neuron_r[j] = np.nan


print()
print("Neuron-wise encoding")
print("-" * 70)

print(
    f"Mean neuron r:   "
    f"{np.nanmean(neuron_r):.4f}"
)

print(
    f"Median neuron r: "
    f"{np.nanmedian(neuron_r):.4f}"
)


# ============================================================
# EVALUATION 3:
# REPRESENTATIONAL GEOMETRY
# ============================================================

R_real = np.corrcoef(Y)
R_pred = np.corrcoef(Y_pred)

iu = np.triu_indices(n_images, k=1)

r_geometry, p_geometry = pearsonr(
    R_real[iu],
    R_pred[iu]
)

print()
print("Representational geometry")
print("-" * 70)

print(
    f"RSM correlation: "
    f"r={r_geometry:.4f}, "
    f"p={p_geometry:.3e}"
)


# ============================================================
# SAVE CROSS-VALIDATED PREDICTIONS
# ============================================================

np.save(
    "vit_to_neural_LOO_predictions.npy",
    Y_pred
)

print("\nSaved predictions.")

ViT: (118, 1000)
Neural: (118, 40064)
Device: cuda
LOO 118/118
Done.

HELD-OUT IMAGE ENCODING
Mean population r:   0.4492
Median population r: 0.4366

Neuron-wise encoding
----------------------------------------------------------------------
Mean neuron r:   0.0185
Median neuron r: 0.0127

Representational geometry
----------------------------------------------------------------------
RSM correlation: r=0.1764, p=2.262e-49

Saved predictions.


In [15]:
Y_mean_pred = np.zeros_like(Y)

for test_idx in range(len(Y)):

    train_mask = np.arange(len(Y)) != test_idx

    Y_mean_pred[test_idx] = Y[train_mask].mean(axis=0)


# ------------------------------------------------------------
# IMAGE-WISE POPULATION CORRELATION
# ------------------------------------------------------------

mean_image_r = []

for i in range(len(Y)):

    r = np.corrcoef(
        Y[i],
        Y_mean_pred[i]
    )[0, 1]

    mean_image_r.append(r)

print("Mean-only population r:",
      np.mean(mean_image_r))


# ------------------------------------------------------------
# NEURON-WISE CORRELATION
# ------------------------------------------------------------

mean_neuron_r = []

for j in range(Y.shape[1]):

    if (
        np.std(Y[:, j]) > 0
        and np.std(Y_mean_pred[:, j]) > 0
    ):

        r = np.corrcoef(
            Y[:, j],
            Y_mean_pred[:, j]
        )[0, 1]

        mean_neuron_r.append(r)

print("Mean-only neuron r:",
      np.mean(mean_neuron_r))


# ------------------------------------------------------------
# RSM
# ------------------------------------------------------------

R_real = np.corrcoef(Y)
R_mean = np.corrcoef(Y_mean_pred)

iu = np.triu_indices(len(Y), k=1)

r = np.corrcoef(
    R_real[iu],
    R_mean[iu]
)[0, 1]

print("Mean-only RSM r:", r)

Mean-only population r: 0.5556555603496318
Mean-only neuron r: -0.9999999999904405
Mean-only RSM r: 0.8199410769839716
